In [ ]:
# Bibliotecas utilizadas
import os
import pandas as pd
import pyodbc
from datetime import datetime
import warnings

# Pega usuário de rede
usuario = os.getenv('USERNAME')

# Tira mensagens de warning
warnings.filterwarnings('ignore')

# Caminho para o arquivo com credenciais
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)

# Carrega credenciais
df_senhas = pd.read_excel('SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

# Conexão com o SQL Server
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Query SQL
query_table = """
SELECT 
    FT.AnoMesRef,
    FT.ID_Produto,
    FT.ID_Comissionado,
    FT.ID_UF,
    FT.ID_Cota,
    FT.ST_Contrato,
    FT.Tipo_Pessoa,
    FT.Grupo,
    FT.Cota,
    FT.Faixa_Atraso,
    FT.Desclassificado,
    FT.TP_Pagto_DemaisParcelas,
    FT.ST_Adimplencia,
    DP.CD_InscricaoNacional,
    VA.DT_EntregaBem,
    CO.CANAL_VENDA,
    VB.DT_Contemplacao,

    CASE 
        WHEN VB.DT_Contemplacao IS NULL OR VA.DT_EntregaBem IS NULL THEN NULL
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 31 THEN 'Over 30'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 61 THEN 'Over 60'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 91 THEN 'Over 90'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 121 THEN 'Over 120'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 151 THEN 'Over 150'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 181 THEN 'Over 180'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 211 THEN 'Over 210'
        ELSE 'Over > 210'
    END AS Categoria_Tempo_Entrega

FROM 
    FT0015_CarteiraCotas AS FT
LEFT JOIN 
    DM0013_Pessoas AS DP ON FT.ID_Pessoa = DP.ID_Pessoa
LEFT JOIN 
    FT0002_VendaAlocacoes AS VA ON FT.ID_Cota = VA.ID_Cota
LEFT JOIN 
    DM0004_Comissionados AS CO ON FT.ID_Comissionado = CO.ID_Comissionado
LEFT JOIN 
    FT0018_Contemplacao AS VB ON FT.ID_Cota = VB.ID_Cota
WHERE
    FT.AnoMesRef = '202502'
"""

# Executa a consulta
print("📥 Executando a consulta SQL...")
df = pd.read_sql(query_table, conn)

# Converte colunas de data
df['DT_EntregaBem'] = pd.to_datetime(df['DT_EntregaBem'], errors='coerce')
df['DT_Contemplacao'] = pd.to_datetime(df['DT_Contemplacao'], errors='coerce')

# Caminho de saída para arquivos
pasta_destino = rf'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos'
csv_path = os.path.join(pasta_destino, 'dados_completos_query.csv')

# Salva os dados brutos como CSV
df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
print(f"✅ Dados exportados com sucesso em CSV:\n{csv_path}")

# === Construção da matriz ===

# Adiciona a coluna 'Safra Contemplacao' no formato 'YYYYMM'
df['Safra Contemplacao'] = df['DT_Contemplacao'].dt.strftime('%Y%m')

# Mapeamento de categorias para colunas M1...M8
mapa_categorias = {
    'Over 30': 'M1',
    'Over 60': 'M2',
    'Over 90': 'M3',
    'Over 120': 'M4',
    'Over 150': 'M5',
    'Over 180': 'M6',
    'Over 210': 'M7',
    'Over > 210': 'M8'
}

# Substitui valores da Categoria_Tempo_Entrega por M1...M8
df['Categoria_M'] = df['Categoria_Tempo_Entrega'].map(mapa_categorias)

# Remove linhas com Categoria_M ou Safra nulas
df_matriz = df.dropna(subset=['Categoria_M', 'Safra Contemplacao'])

# Agrupa e conta CD_InscricaoNacional distintos por Safra e Categoria_M
matriz = (
    df_matriz.groupby(['Safra Contemplacao', 'Categoria_M'])['CD_InscricaoNacional']
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
    .rename_axis(None, axis=1)
)

# Ordena colunas: Safra primeiro, depois M1...M8
colunas_ordenadas = ['Safra Contemplacao'] + [f'M{i}' for i in range(1, 9)]
matriz = matriz.reindex(columns=colunas_ordenadas)

# Salva a matriz como CSV
matriz_path = os.path.join(pasta_destino, 'matriz_tempo_entrega_2.csv')
matriz.to_csv(matriz_path, sep=';', index=False, encoding='utf-8-sig')
print(f"✅ Matriz exportada com sucesso em CSV:\n{matriz_path}")


📥 Executando a consulta SQL...
✅ Dados exportados com sucesso em CSV:
C:\Users\GabrielHenriqueSilva\OneDrive - CAIXA Consórcio\Documentos\dados_completos_query.csv
✅ Matriz exportada com sucesso em CSV:
C:\Users\GabrielHenriqueSilva\OneDrive - CAIXA Consórcio\Documentos\matriz_tempo_entrega.csv
